# Learned g — CPU Preparation

**Primary author:** Victoria Winters

**Builds on:**
- *02_embedding_generation.ipynb* (Victoria — stock CALE embeddings and index files)
- *05_dataset_construction.ipynb* (Victoria — `dataset_harder.parquet` with matched real/distractor pairs)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

This notebook prepares all inputs needed by the GPU training script
(`scripts/train_g_triplet.py`) for Phase 2's learned g experiment.
It loads the Phase 1 harder dataset and stock CALE embeddings, performs
a train/test split following Egami et al. (2022), constructs training
triplets for triplet loss, and extracts test-set embeddings — all on
CPU, with no model loading or GPU work.

**Experimental question (from PHASE2_CONTEXT.md):** Can we learn a
custom codebook function g by fine-tuning CALE on cryptic crossword
triplets, and does the resulting embedding space reduce the misdirection
ATE compared to stock CALE?

**Inputs:** `data/dataset_harder.parquet`, Phase 1 embedding `.npy`
files and index CSVs from `data/embeddings/`.

**Outputs:** `data/learned_g/{DATASET}/` — split config, training
triplets, test-set stock embeddings. See DATA.md for the full schema.

In [ ]:
import json
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

## Configuration

Change `DATASET` to rerun this notebook for a different distractor set
(e.g., `'easy'` or a future WordNet-based dataset). All downstream paths
are derived from this variable, so no other code changes are needed.

Set `SAMPLE_MODE = True` for a fast end-to-end test before running on
the full dataset.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────
DATASET = 'harder'        # which dataset to load and where outputs are saved
RANDOM_SEED = 42
TEST_FRACTION = 0.5       # held out until final ATE estimation only;
                          # never used to compare or select g's
VAL_FRACTION  = 0.2       # used to compare different g's and select
                          # the best one before touching the test set
SAMPLE_MODE = False       # set True for fast testing (5000 rows)
SAMPLE_SIZE = 5000

# ── Paths (derived from DATASET) ────────────────────────────────────
DATA_DIR = Path('..') / 'data'
EMB_DIR = DATA_DIR / 'embeddings'
OUTPUT_DIR = DATA_DIR / 'learned_g' / DATASET

## Section 1 — Load dataset and split

### Why a three-way split, and why group by def_answer_pair_id?

The Egami et al. (2022) framework requires a strict train/test
separation: the codebook function g is learned on the training set,
then *locked* before being applied to the test set for ATE estimation.
But in practice we will train multiple versions of g — varying
hyperparameters (learning rate, margin, epochs) and potentially
distractor construction strategies. If we used the test set to compare
these candidates, every comparison would leak information from the test
set into our model selection, inflating the final ATE estimate.

To prevent this, we hold out a **validation set** between train and
test. The workflow is:

1. **Train** various g's on the training set.
2. **Compare** them on the validation set (e.g., by validation triplet
   loss, or by preliminary ATE on validation data).
3. **Select** a single g based on validation performance.
4. **Lock** that g and evaluate it exactly once on the test set to
   produce the final ATE estimate reported in the paper.

The test set is saved now but must not be examined or used for any
decision until step 4. This discipline is what makes the final ATE
estimate credible.

We split only the **real** rows (label == 1) and group by
`def_answer_pair_id` so that all rows involving the same
definition–answer pair land in the same split. This is the same
leakage-prevention strategy used in Phase 1's GroupKFold
cross-validation (see NB06/NB07). We do not split distractor rows
independently — each distractor is matched 1:1 to a real row by
`(clue_id, definition_wn)`, so its split assignment follows from its
matched real row.

The split sizes (50% test, 20% validation, 30% train) follow Egami
et al.'s recommendation for a large test set. The training set is
smaller than typical ML setups, but triplet loss with ~70K triplets
is sufficient for fine-tuning a pretrained model.

In [ ]:
# Load the labeled dataset (real pairs + distractors)
df = pd.read_parquet(DATA_DIR / f'dataset_{DATASET}.parquet')
print(f"Loaded dataset_{DATASET}.parquet: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"  Real rows (label=1): {(df['label']==1).sum():,}")
print(f"  Distractor rows (label=0): {(df['label']==0).sum():,}")

# In SAMPLE_MODE, take a random subset of real rows and their matched distractors
if SAMPLE_MODE:
    real_sample = df[df['label'] == 1].sample(
        n=min(SAMPLE_SIZE, (df['label'] == 1).sum()),
        random_state=RANDOM_SEED
    )
    # Find matched distractors by (clue_id, definition_wn)
    keys = real_sample[['clue_id', 'definition_wn']].drop_duplicates()
    dist_sample = df[df['label'] == 0].merge(keys, on=['clue_id', 'definition_wn'])
    df = pd.concat([real_sample, dist_sample], ignore_index=True)
    print(f"\nSAMPLE_MODE: reduced to {df.shape[0]:,} rows "
          f"({(df['label']==1).sum():,} real + {(df['label']==0).sum():,} distractors)")

In [ ]:
# Train/val/test split on real rows only, grouped by def_answer_pair_id.
# Two-stage split:
#   Stage 1: test (TEST_FRACTION) vs. remainder — RANDOM_SEED
#   Stage 2: remainder → train vs. val — RANDOM_SEED + 1
real_mask = df['label'] == 1
real_df = df[real_mask].copy()
groups = real_df['def_answer_pair_id'].values

# Stage 1: split off the test set
gss_test = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=RANDOM_SEED)
remainder_idx, test_idx = next(gss_test.split(real_df, groups=groups))

# Stage 2: split the remainder into train and validation.
# VAL_FRACTION is expressed as a fraction of the *total*, so within the
# remainder it is VAL_FRACTION / (1 - TEST_FRACTION).
val_frac_of_remainder = VAL_FRACTION / (1 - TEST_FRACTION)
remainder_df = real_df.iloc[remainder_idx]
remainder_groups = remainder_df['def_answer_pair_id'].values

gss_val = GroupShuffleSplit(n_splits=1, test_size=val_frac_of_remainder, random_state=RANDOM_SEED + 1)
train_idx_in_rem, val_idx_in_rem = next(gss_val.split(remainder_df, groups=remainder_groups))

# Map back to original dataframe indices
real_indices = real_df.index.values
train_real_indices = set(real_indices[remainder_idx[train_idx_in_rem]])
val_real_indices = set(real_indices[remainder_idx[val_idx_in_rem]])
test_real_indices = set(real_indices[test_idx])

# Tag every row in the dataframe with its split assignment
df['split'] = 'none'
df.loc[df.index.isin(train_real_indices), 'split'] = 'train'
df.loc[df.index.isin(val_real_indices), 'split'] = 'val'
df.loc[df.index.isin(test_real_indices), 'split'] = 'test'

print(f"Train real rows: {len(train_real_indices):,}")
print(f"Val   real rows: {len(val_real_indices):,}")
print(f"Test  real rows: {len(test_real_indices):,}")

train_groups = set(real_df.loc[list(train_real_indices), 'def_answer_pair_id'])
val_groups = set(real_df.loc[list(val_real_indices), 'def_answer_pair_id'])
test_groups = set(real_df.loc[list(test_real_indices), 'def_answer_pair_id'])

print(f"\nTrain def_answer_pair_id groups: {len(train_groups):,}")
print(f"Val   def_answer_pair_id groups: {len(val_groups):,}")
print(f"Test  def_answer_pair_id groups: {len(test_groups):,}")

# Verify no group leakage across any pair of splits
assert len(train_groups & val_groups) == 0, "Train/val group leakage!"
assert len(train_groups & test_groups) == 0, "Train/test group leakage!"
assert len(val_groups & test_groups) == 0, "Val/test group leakage!"
print("\nNo group leakage across train/val/test splits. ✓")

## Section 2 — Build index lookups

### Why a composite key for clue-context embeddings?

The clue-context embedding array (`clue_context_embeddings.npy`) has one
row per clue–definition combination. Most clues have a single definition,
so `clue_id` alone would be sufficient to look up the embedding. But
**double-definition clues** have two rows with the same `clue_id` (one
per definition), each with a different embedding because the `<t></t>`
delimiters surround a different word in the surface text. Using
`clue_id` alone would silently return the wrong embedding for one of
the two definitions. The composite key `(clue_id, definition_wn)`
uniquely identifies each row.

We build the composite-key lookup from `clue_context_phrases.csv`
(which has both `clue_id` and `definition_wn` columns) rather than
from `clue_context_index.csv` (which only has `clue_id`), following
the convention in CLAUDE.md.

In [ ]:
# Load index CSVs — always use keep_default_na=False because "nan"
# (meaning grandmother) is a valid crossword word
def_index = pd.read_csv(EMB_DIR / 'definition_index.csv', keep_default_na=False)
ans_index = pd.read_csv(EMB_DIR / 'answer_index.csv', keep_default_na=False)
clue_phrases = pd.read_csv(EMB_DIR / 'clue_context_phrases.csv', keep_default_na=False)

# definition_wn string → row position in definition_embeddings.npy
def_to_idx = {row['word']: idx for idx, row in def_index.iterrows()}

# answer_wn string → row position in answer_embeddings.npy
ans_to_idx = {row['word']: idx for idx, row in ans_index.iterrows()}

# (clue_id, definition_wn) → row position in clue_context_embeddings.npy
clue_ctx_to_idx = {
    (row['clue_id'], row['definition_wn']): idx
    for idx, row in clue_phrases.iterrows()
}

print(f"Definition lookup: {len(def_to_idx):,} unique words")
print(f"Answer lookup:     {len(ans_to_idx):,} unique words")
print(f"Clue-context lookup: {len(clue_ctx_to_idx):,} (clue_id, definition_wn) pairs")

In [ ]:
# Load embedding arrays (memory-mapped for efficiency — we only read
# specific rows, not the entire array into RAM)
def_emb = np.load(EMB_DIR / 'definition_embeddings.npy', mmap_mode='r')
ans_emb = np.load(EMB_DIR / 'answer_embeddings.npy', mmap_mode='r')
clue_ctx_emb = np.load(EMB_DIR / 'clue_context_embeddings.npy', mmap_mode='r')

print(f"definition_embeddings.npy: {def_emb.shape}")
print(f"answer_embeddings.npy:     {ans_emb.shape}")
print(f"clue_context_embeddings.npy: {clue_ctx_emb.shape}")

## Section 3 — Construct training triplets

### What is a triplet and why allsense embeddings?

**Triplet loss** is a metric learning objective that teaches an embedding
model to place semantically related items closer together and unrelated
items farther apart. Each training example is a triplet of three
embeddings:

- **Anchor:** the definition word (what we're trying to match)
- **Positive:** the true answer (what should be close to the anchor)
- **Negative:** a distractor answer (what should be far from the anchor)

The loss penalizes the model when the negative is closer to the anchor
than the positive (plus a margin). Over many triplets, the model learns
an embedding space where true definition–answer relationships are
tighter than spurious ones — i.e., a space that partially resists the
misdirection introduced by cryptic clue context.

We use **allsense-average embeddings** (slot 0) for all three
components rather than clue-context embeddings for the anchor. This is
deliberate: the goal of learning g is to improve the *context-free*
embedding space so that definitions and their true answers are
intrinsically closer together, independent of any particular clue's
surface reading. Using clue-context embeddings as anchors would train
the model to match a *specific misleading context* to its answer,
which is the opposite of what we want. The clue-context embeddings
are reserved for the ATE estimation step (on the test set), where we
compare retrieval with and without context under the learned g.

In [ ]:
# Build a lookup from (clue_id, definition_wn) → distractor row for
# fast matching. Each real row has exactly one matched distractor in the
# harder dataset (1:1 by construction in NB05).
dist_df = df[df['label'] == 0]
dist_lookup = {
    (row['clue_id'], row['definition_wn']): row
    for _, row in dist_df.iterrows()
}

# Iterate over training real rows and construct triplets
train_real = df[(df['split'] == 'train') & (df['label'] == 1)]

anchors = []
positives = []
negatives = []
skipped = 0

for _, row in train_real.iterrows():
    defn = row['definition_wn']
    ans = row['answer_wn']
    key = (row['clue_id'], row['definition_wn'])

    # Look up the matched distractor
    dist_row = dist_lookup.get(key)
    if dist_row is None:
        skipped += 1
        continue

    distractor_word = dist_row['distractor_source']

    # All three lookups must succeed
    if defn not in def_to_idx or ans not in ans_to_idx or distractor_word not in ans_to_idx:
        skipped += 1
        continue

    # Slot 0 = allsense_avg embedding
    anchors.append(def_emb[def_to_idx[defn], 0, :])
    positives.append(ans_emb[ans_to_idx[ans], 0, :])
    negatives.append(ans_emb[ans_to_idx[distractor_word], 0, :])

# Stack into arrays
anchors = np.array(anchors, dtype=np.float32)
positives = np.array(positives, dtype=np.float32)
negatives = np.array(negatives, dtype=np.float32)

print(f"Training triplets constructed: {len(anchors):,}")
print(f"Real rows skipped (missing distractor or embedding): {skipped:,}")
print(f"Triplet shapes: anchors {anchors.shape}, positives {positives.shape}, negatives {negatives.shape}")

## Section 4 — Extract validation and test stock embeddings

### Why extract now on CPU instead of re-embedding on GPU?

We already have stock CALE embeddings for every definition, answer, and
clue-context phrase from Phase 1 (NB02). Re-computing these on the GPU
would waste compute time and introduce a risk of inconsistency — any
difference in tokenization, batching, or library version could produce
slightly different vectors, making the stock-vs-learned comparison
invalid. By slicing directly from the Phase 1 `.npy` files on CPU, we
guarantee that the stock embeddings used in the ATE comparison are
*exactly* the same ones used in Phase 1's retrieval analysis.

We extract stock embeddings for both the **validation** and **test**
sets here. The validation embeddings will be used freely during model
selection — comparing different g's by their validation ATE or triplet
loss. The test embeddings are saved to disk now but **must not be
examined or used for any decision** until a single g has been finalized
on the validation set. Only then does the GPU training script load
the test embeddings and compute the final ATE estimate.

The GPU training script (`train_g_triplet.py`) will re-embed both sets
with the *learned* g to produce learned embeddings. The stock embeddings
extracted here serve as the baseline for those comparisons.

In [ ]:
def extract_stock_embeddings(subset_df, split_name):
    """Extract stock CALE embeddings for a subset of real rows.

    Returns (def_arr, ans_arr, clue_arr, indices) or raises if nothing
    could be extracted.
    """
    def_list, ans_list, clue_list, idx_list = [], [], [], []
    skipped = 0

    for orig_idx, row in subset_df.iterrows():
        defn = row['definition_wn']
        ans = row['answer_wn']
        clue_key = (row['clue_id'], row['definition_wn'])

        if (defn not in def_to_idx or ans not in ans_to_idx
                or clue_key not in clue_ctx_to_idx):
            skipped += 1
            continue

        def_list.append(def_emb[def_to_idx[defn], 0, :])       # allsense
        ans_list.append(ans_emb[ans_to_idx[ans], 0, :])         # allsense
        clue_list.append(clue_ctx_emb[clue_ctx_to_idx[clue_key]])
        idx_list.append(orig_idx)

    print(f"{split_name}: {len(idx_list):,} rows extracted, {skipped:,} skipped")
    return (
        np.array(def_list, dtype=np.float32),
        np.array(ans_list, dtype=np.float32),
        np.array(clue_list, dtype=np.float32),
        np.array(idx_list, dtype=np.int64),
    )

# Validation set
val_real = df[(df['split'] == 'val') & (df['label'] == 1)]
val_stock_def, val_stock_ans, val_stock_clue, val_indices = \
    extract_stock_embeddings(val_real, 'Validation')

# Test set
test_real = df[(df['split'] == 'test') & (df['label'] == 1)]
test_stock_def, test_stock_ans, test_stock_clue, test_indices = \
    extract_stock_embeddings(test_real, 'Test')

print(f"\nVal  shapes: def {val_stock_def.shape}, ans {val_stock_ans.shape}, "
      f"clue {val_stock_clue.shape}")
print(f"Test shapes: def {test_stock_def.shape}, ans {test_stock_ans.shape}, "
      f"clue {test_stock_clue.shape}")

## Section 5 — Save outputs

All outputs go to `data/learned_g/{DATASET}/` following the schema
defined in DATA.md.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. split_config.json — records all parameters for reproducibility
split_config = {
    'dataset': DATASET,
    'random_seed': RANDOM_SEED,
    'test_fraction': TEST_FRACTION,
    'val_fraction': VAL_FRACTION,
    'sample_mode': SAMPLE_MODE,
    'sample_size': SAMPLE_SIZE if SAMPLE_MODE else None,
    'n_train_triplets': int(len(anchors)),
    'n_val_rows': int(len(val_indices)),
    'n_test_rows': int(len(test_indices)),
    'date_created': str(datetime.date.today()),
}
with open(OUTPUT_DIR / 'split_config.json', 'w') as f:
    json.dump(split_config, f, indent=2)

# 2. Index arrays — row indices into the original dataframe
np.save(OUTPUT_DIR / 'val_indices.npy', val_indices)
np.save(OUTPUT_DIR / 'test_indices.npy', test_indices)

# 3. train_triplets.npz — compressed archive with anchors, positives, negatives
np.savez_compressed(
    OUTPUT_DIR / 'train_triplets.npz',
    anchors=anchors,
    positives=positives,
    negatives=negatives,
)

# 4. Validation stock embeddings
np.save(OUTPUT_DIR / 'val_stock_def_emb.npy', val_stock_def)
np.save(OUTPUT_DIR / 'val_stock_ans_emb.npy', val_stock_ans)
np.save(OUTPUT_DIR / 'val_stock_clue_emb.npy', val_stock_clue)

# 5. Test stock embeddings
np.save(OUTPUT_DIR / 'test_stock_def_emb.npy', test_stock_def)
np.save(OUTPUT_DIR / 'test_stock_ans_emb.npy', test_stock_ans)
np.save(OUTPUT_DIR / 'test_stock_clue_emb.npy', test_stock_clue)

# Report file sizes
print("Saved to:", OUTPUT_DIR)
print()
for p in sorted(OUTPUT_DIR.iterdir()):
    size_mb = p.stat().st_size / (1024 * 1024)
    if p.suffix == '.json':
        print(f"  {p.name:35s}  {p.stat().st_size:>10,} bytes")
    else:
        print(f"  {p.name:35s}  {size_mb:>8.1f} MB")

## Section 6 — Verification

### Why check that positives are more similar to anchors than negatives?

Triplet loss trains the model to push positives closer to anchors than
negatives. For this to be a *learnable* objective, the stock embedding
space should already show this tendency on average — true answers should
already be somewhat more similar to their definitions than distractor
answers are. If this baseline condition fails (negatives are as similar
or more similar to anchors than positives), it means the harder
distractors are already indistinguishable from true answers in stock
CALE space. That would itself be an interesting finding — it would
suggest that the cosine-similarity-based distractors are *too hard* for
the model to learn from, and we might need a different distractor
construction strategy (e.g., WordNet-based).

Note that we expect the gap to be **small** for the harder dataset by
design — these distractors were specifically chosen to be cosine-similar
to the definition. A small but positive gap is the expected starting
point for triplet loss to improve upon.

In [ ]:
# Reload saved triplets to verify round-trip integrity
loaded = np.load(OUTPUT_DIR / 'train_triplets.npz')
assert loaded['anchors'].shape == anchors.shape, "Anchor shape mismatch"
assert loaded['positives'].shape == positives.shape, "Positive shape mismatch"
assert loaded['negatives'].shape == negatives.shape, "Negative shape mismatch"
print(f"Reloaded triplets — shapes match: {loaded['anchors'].shape}")

# Cosine similarity helper (vectorized, row-wise)
def cosine_sim(a, b):
    """Row-wise cosine similarity between two (N, D) arrays."""
    dot = np.sum(a * b, axis=1)
    norm_a = np.linalg.norm(a, axis=1)
    norm_b = np.linalg.norm(b, axis=1)
    return dot / (norm_a * norm_b + 1e-10)

# Compare anchor-positive vs anchor-negative similarities
sim_pos = cosine_sim(loaded['anchors'], loaded['positives'])
sim_neg = cosine_sim(loaded['anchors'], loaded['negatives'])

print(f"\nMean cosine similarity (anchor ↔ positive): {sim_pos.mean():.4f} ± {sim_pos.std():.4f}")
print(f"Mean cosine similarity (anchor ↔ negative): {sim_neg.mean():.4f} ± {sim_neg.std():.4f}")
print(f"Gap (positive - negative): {(sim_pos - sim_neg).mean():.4f}")

frac_correct = (sim_pos > sim_neg).mean()
print(f"Fraction where positive > negative: {frac_correct:.3f}")

if sim_pos.mean() > sim_neg.mean():
    print("\n✓ Positives are more similar to anchors on average — triplet loss has a learnable signal.")
else:
    print("\n⚠ WARNING: Negatives are as similar or MORE similar than positives. "
          "This may indicate distractors are too hard for triplet loss to learn from.")

In [ ]:
# Test-set vocabulary coverage
test_subset = df.loc[test_indices]
n_unique_defs = test_subset['definition_wn'].nunique()
n_unique_ans = test_subset['answer_wn'].nunique()
print(f"Test set: {len(test_indices):,} rows")
print(f"  Unique definition words: {n_unique_defs:,}")
print(f"  Unique answer words:     {n_unique_ans:,}")

## Summary

This notebook prepared all inputs for the GPU training script
(`scripts/train_g_triplet.py`).

**What was produced:**
- **Training triplets** (`train_triplets.npz`): anchors, positives, and
  negatives for triplet loss, each shape (N_triplets, 1024). Built from
  allsense-average embeddings (slot 0) of stock CALE.
- **Validation stock embeddings** (`val_stock_*.npy`): definition
  allsense, answer allsense, and clue-context embeddings for the
  validation set, each shape (N_val, 1024). Used freely to compare
  candidate g's.
- **Test stock embeddings** (`test_stock_*.npy`): same three embedding
  types for the held-out test set, each shape (N_test, 1024). **Do not
  use until a single g has been finalized on the validation set.**
- **Split metadata** (`split_config.json`, `val_indices.npy`,
  `test_indices.npy`): everything needed to reproduce or inspect the
  three-way split.

**Intended workflow:**
1. **Train** g on training triplets (GPU script).
2. **Compare** candidate g's on the validation set — evaluate
   validation triplet loss or preliminary ATE using `val_stock_*.npy`
   vs. learned validation embeddings.
3. **Select** a single g based on validation performance.
4. **Report** the final ATE on the test set, using that locked g
   exactly once. This is the number that goes in the paper.

**Key design decisions:**
- **Three-way split** (30% train / 20% val / 50% test) following
  Egami et al.'s recommendation for a large test set, with a validation
  set added for model selection.
- **Grouped by `def_answer_pair_id`** to prevent leakage — no
  definition–answer pair appears in more than one split.
- **Allsense embeddings for triplets** — we train on context-free
  representations so the model learns intrinsic definition–answer
  associations, not context-specific ones.
- **Stock embeddings sliced from Phase 1 files** — no re-computation,
  ensuring exact consistency with Phase 1 baselines.

**Next step:** Run `scripts/train_g_triplet.py` on GPU (Great Lakes)
to fine-tune CALE, re-embed the validation set with the learned g, and
compare candidate g's before touching the test set.